# Results Analysis & Paper Tables
Produces all tables and figures for the paper

In [ ]:
# !pip install ultralytics rich -q

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/rsna-lumbar-yolo')
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json
RUNS_DIR = Path('runs')
OUTPUT_DIR = Path('runs/paper_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.train.train_ablation import collect_results, compute_ablation_summary, print_ablation_table
run_df = collect_results(RUNS_DIR, ['yolo11n', 'yolo11m'])
summary_df = compute_ablation_summary(run_df)
print_ablation_table(summary_df)

In [ ]:
from src.eval.compare_models import (
    load_all_run_results, load_all_eval_results, compute_model_summary,
    build_comparison_table, export_latex_comparison
)
run_df = load_all_run_results(RUNS_DIR, ['yolo11n', 'yolo11m'])
eval_df = load_all_eval_results(RUNS_DIR, ['yolo11n', 'yolo11m'])
summary_df = compute_model_summary(run_df, eval_df)
comp_table = build_comparison_table(summary_df)
from rich.console import Console; from rich.table import Table as RichTable
console = Console()
rtable = RichTable(title="Model Comparison")
for col in comp_table.columns: rtable.add_column(col)
for _, row in comp_table.iterrows(): rtable.add_row(*[str(v) for v in row])
console.print(rtable)
export_latex_comparison(comp_table, OUTPUT_DIR / 'model_comparison.tex')

In [ ]:
from src.eval.compare_models import compute_per_class_delta, plot_metric_comparison
per_class_df = compute_per_class_delta(eval_df)
display(per_class_df)
plot_metric_comparison(summary_df, per_class_df, OUTPUT_DIR)
from IPython.display import Image
Image(str(OUTPUT_DIR / 'per_class_delta.png'))

In [ ]:
from src.eval.per_level_table import load_detections, load_ground_truth_long, compute_per_column_metrics, build_paper_table, export_latex_table
# Load detections from best model run
detections_json = RUNS_DIR / 'detections.json'  # produced by compute_metrics main
if detections_json.exists():
    det_df = load_detections(detections_json)
    gt_long = load_ground_truth_long(Path('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train.csv'))
    metrics_df = compute_per_column_metrics(det_df, gt_long)
    paper_df = build_paper_table(metrics_df)
    paper_df.to_csv(OUTPUT_DIR / 'paper_table.csv', index=False)
    export_latex_table(paper_df, OUTPUT_DIR / 'paper_table.tex')
    console.print(paper_df.to_string())
else:
    print("detections.json not found — run compute_metrics.py first")

In [ ]:
print("PR curves: requires detections.json with confidence scores")
print("Run: python -m src.eval.compute_metrics --weights runs/yolo11m_fold0/weights/best.pt ...")

## Paper Results Summary

Fill in after training completes:

- Best model: [yolo11n / yolo11m]
- Overall mAP@50: ___
- Overall RSNA weighted log-loss: ___ (uniform baseline: 1.099)
- Improvement over uniform baseline: ___

**Abstract snippet:**
"We trained a YOLOv11 detector on the RSNA 2024 Lumbar Spine dataset
achieving mAP@50 of ___ and RSNA weighted log-loss of ___,
compared to the uniform-prediction baseline of 1.099."

In [ ]:
print("05_results_analysis.ipynb Complete ✓")